# The Price is Right — Week 7 Excercise
## This excercise is Fine-Tuning Qwen2.5-3B for Product Price Prediction

**Student:** Geraldino07  
**Date:** March 2026  

---


## Project Description

This is my Week 7 independent project, built on top of everything Ed taught us about fine-tuning open source LLMs with QLoRA.

Rather than simply replicating Ed's results with `meta-llama/Llama-3.2-3B`, I decided to test a hypothesis:

> *Would a model with a slightly larger vocabulary tokenise product descriptions more efficiently — and therefore learn better from the same training data?*

To test this, I swapped in **Alibaba's Qwen2.5-3B** as the base model, keeping every other variable identical — same dataset, same QLoRA setup, same training infrastructure. **And it did!!.**

---

### Why Qwen2.5-3B?

| | Llama-3.2-3B | Qwen2.5-3B |
|---|---|---|
| Vocabulary size | ~128,256 tokens | ~151,936 tokens |
| Licence | Gated (Meta) | Apache 2.0 |
| Parameter count | 3B | 3B |
| Avg Error (2 epochs, lite) | $65.40 | **$51.45** |





---

### Technical Approach

Following Ed's QLoRA methodology exactly:

- **Base model**: `Qwen/Qwen2.5-3B`
- **Dataset**: `ed-donner/items_prompts_lite` (20K examples)
- **Quantisation**: 4-bit NF4 (QLoRA) with double quantisation
- **LoRA rank**: r=32, alpha=64
- **Target modules**: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`
- **Optimiser**: `paged_adamw_8bit`
- **Learning rate**: 2e-4 with cosine schedule
- **Effective batch size**: 16 (batch 4 × gradient accumulation 4)
- **Epochs**: 2


## 🏆 Results

| Model | Avg Error | MSE | R² |
|---|---|---|---|
| Human Baseline | $87.62 | — | — |
| Ed's Llama-3.2-3B (lite) | $65.40 | 13,751 | 37.4% |
| GPT-4.1-nano (reference) | $62.51 | — | — |
| Qwen2.5-3B — Epoch 1 | $63.13 | 11,168 | 49.2% |
| **Qwen2.5-3B — Epoch 2** | **$51.45** | **7,701** | **65.0%** |

### Key Findings

- ✅ **Beat the human baseline** by $36.17
- ✅ **Beat GPT-4.1-nano** by $11.06 — a large commercial model — using a H100 GPU
- ✅ **Beat Ed's Llama-3.2-3B** (same lite dataset) by $13.95
- ✅ **R² of 65.0%** — up from 37.4% with Llama, meaning the model explains 65% of price variance
- ✅ **Second epoch was critical** — $11.68 improvement from epoch 1 to epoch 2

### Training Loss

```
Epoch 1 Start  (Step  250)  →  1.037
Epoch 1 End    (Step 1500)  →  0.804
Epoch 2 Mid    (Step 2175)  →  0.694
Epoch 2 End    (Step 2500)  →  0.712
```
Consistent downward trend confirmed the model kept learning through both epochs.


## Links

### Full Google Colab Notebook
The complete notebook — including all code, training output, loss curves, and evaluation charts — is available here:

https://colab.research.google.com/drive/1pL9D4beB57TWT9uBTjbsa5TGeR6Xf8OI#scrollTo=reflection_header

---

### Fine-Tuned Model on HuggingFace
The trained LoRA adapter (epoch 2) is publicly available on HuggingFace Hub:

https://huggingface.co/Geraldino07/price-2026-03-09_21.46.47

```python

BASE_MODEL = "Qwen/Qwen2.5-3B"
HUB_MODEL  = "Geraldino07/price-2026-03-09_21.46.47"

